In [1]:
from torchvision import datasets, transforms

# load dataset
train_set = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

# use DataLoader to create batches of data
from torch.utils.data import DataLoader
batch_size = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

# check the batch
batch_images, batch_labels = next(iter(train_loader))
print(batch_images.shape)
print(batch_labels.shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [2]:
# create model
import torch.nn as nn

model = nn.Linear(28 * 28, 10)
print(sum(p.numel() for p in model.parameters()))

7850


In [ ]:
# Write the full training loop: get batch → flatten → forward → loss → backward → optimizer step.
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# process one batch
images, labels = next(iter(train_loader))
print(images.shape)
flattened_images = images.view(-1, 28 * 28)
logits = model(flattened_images)
loss = criterion(logits, labels)
print(f"Loss: {loss.item()}")

# zero the gradients
optimizer.zero_grad()

# before backward
print("grad before backward:", model.weight.grad)   # None on first call
loss.backward()

# after backward, before step
print("grad shape:", model.weight.grad.shape)
print("grad norm:", model.weight.grad.norm().item())
old_weight = model.weight[0, 0].item()

optimizer.step()

# after step
new_weight = model.weight[0, 0].item()
print(f"weight[0,0]: {old_weight:.6f} -> {new_weight:.6f}")
# the change should equal -lr * grad[0,0]

In [ ]:
# the loop
running_loss = 0.0
model.train() # no-op now
for i, (images, labels) in enumerate(train_loader):

    flattened_images = images.view(-1, 28 * 28)
    logits = model(flattened_images)
    loss = criterion(logits, labels)

    if i == 0:
        running_loss = loss.item()
    else:
        running_loss = running_loss * 0.9 + loss.item() * 0.1
        
    if i % 100 == 0:
        print(f"step {i}  loss={loss.item():.3f}  ema={running_loss:.3f}")

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
